# Comparing the N2560 RAL3.3 simulations: control vs +4K SST

* Compares `um_glm_n2560_RAL3p3_tuned_hk26` (control) with `um_glm_n2560_RAL3p3_tuned_p4k_hk26`,
  an otherwise identical simulation with sea surface temperatures raised by 4K.
* Same model, grid, period (2020-01-20 to 2021-03-01) and output, so differences are the response to the
  warmer ocean.
* Everything here is done at modest zoom levels so it runs in a few minutes; raise `ZOOM` if you want detail.


## Setup


In [ ]:
import cartopy.crs as ccrs
import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import easygems.healpix as egh

from utils import hp_mods

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
CONTROL = "um_glm_n2560_RAL3p3_tuned_hk26"
P4K = "um_glm_n2560_RAL3p3_tuned_p4k_hk26"

# Zoom for the maps: 7 is ~50 km and loads quickly. 8-9 look better but take longer.
ZOOM = 7
# Zoom for zonal means (needs enough cells per 1-degree bin).
ZONAL_ZOOM = 6
# A day to look at. 2020-07-01 is mid-year, so well away from any spin-up.
DAY = "2020-07-01"

In [ ]:
url = "https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml"
cat = intake.open_catalog(url)["UK"]
# Use online if not on JASMIN:
# cat = intake.open_catalog(url)["online"]

if P4K not in cat:
    # The +4K entry is still in review (catalog PR). Until it is merged, read the local clone.
    cat = intake.open_catalog("/home/users/mmuetz/projects/external/catalog/UK/main.yaml")

print("\n".join(k for k in cat if k.endswith("_hk26")))

In [ ]:
ds1h = {name: cat[sim](zoom=ZOOM, time="PT1H").to_dask().pipe(hp_mods)
        for name, sim in [("control", CONTROL), ("+4K", P4K)]}
ds3h = {name: cat[sim](zoom=ZOOM, time="PT3H").to_dask().pipe(hp_mods)
        for name, sim in [("control", CONTROL), ("+4K", P4K)]}
ds1h["control"]

## Side-by-side maps

Control and +4K on a shared colour scale, with their difference alongside. A shared scale matters: plotted
separately, two fields that differ by a few K look identical.


In [ ]:
def compare_maps(da_ctrl, da_p4k, title, cmap="viridis", diff_cmap="bwr", diff_max=None, **kwargs):
    """Control, +4K and (+4K - control) on one row, the first two sharing a colour scale."""
    vmin = float(min(da_ctrl.min(), da_p4k.min()))
    vmax = float(max(da_ctrl.max(), da_p4k.max()))
    diff = da_p4k - da_ctrl
    if diff_max is None:
        diff_max = float(np.nanpercentile(np.abs(diff.values), 99))

    projection = ccrs.Robinson(central_longitude=0)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4), subplot_kw={"projection": projection},
                             layout="constrained")
    for ax, da, name, kw in [
        (axes[0], da_ctrl, "control", dict(cmap=cmap, vmin=vmin, vmax=vmax)),
        (axes[1], da_p4k, "+4K", dict(cmap=cmap, vmin=vmin, vmax=vmax)),
        (axes[2], diff, "+4K - control", dict(cmap=diff_cmap, vmin=-diff_max, vmax=diff_max)),
    ]:
        ax.set_global()
        im = egh.healpix_show(da, ax=ax, **{**kw, **kwargs})
        ax.coastlines(linewidth=0.5)
        ax.set_title(f"{title}: {name}")
        plt.colorbar(im, ax=ax, orientation="horizontal", shrink=0.8,
                     label=da_ctrl.attrs.get("units", ""))
    return fig

In [ ]:
# Near-surface temperature at midday. The +4K signal should be obvious over ocean, muted over land.
time = pd.Timestamp(f"{DAY} 12:00")
tas = {k: ds.tas.sel(time=time).compute() for k, ds in ds1h.items()}
compare_maps(tas["control"], tas["+4K"], "tas (K)", cmap="RdYlBu_r");

In [ ]:
# Precipitation. Instantaneous rain is noisy, so use a daily mean.
pr = {k: ds.pr.sel(time=DAY).mean(dim="time").compute() * 86400 for k, ds in ds1h.items()}
for k in pr:
    pr[k].attrs["units"] = "mm day-1"
compare_maps(pr["control"], pr["+4K"], f"precipitation, {DAY} mean (mm/day)", cmap="Blues");

In [ ]:
# Outgoing longwave radiation: a good proxy for deep convection (low OLR = high cold cloud).
rlut = {k: ds.rlut.sel(time=DAY).mean(dim="time").compute() for k, ds in ds1h.items()}
compare_maps(rlut["control"], rlut["+4K"], f"rlut, {DAY} mean (W/m2)", cmap="gray_r");

In [ ]:
# Total cloud fraction.
clt = {k: ds.clt.sel(time=DAY).mean(dim="time").compute() for k, ds in ds1h.items()}
compare_maps(clt["control"], clt["+4K"], f"clt, {DAY} mean (%)", cmap="Greys_r");

## Zonal means

Where the differences sit in latitude: does the ITCZ shift, does the precipitation maximum sharpen?


In [ ]:
def zonal_mean(da, bins=np.linspace(-90, 90, 181)):
    dims = [d for d in ("time",) if d in da.dims]
    da = da.mean(dim=dims) if dims else da
    return da.groupby_bins("lat", bins).mean(), (bins[:-1] + bins[1:]) / 2


def compare_zonal(var, scale=1.0, units=None, days=7):
    """Zonal mean of the first `days` of DAY for both sims, plus the difference."""
    times = slice(DAY, str(pd.Timestamp(DAY) + pd.Timedelta(days=days - 1)))
    das = {}
    for name, sim in [("control", CONTROL), ("+4K", P4K)]:
        ds = cat[sim](zoom=ZONAL_ZOOM, time="PT1H").to_dask().pipe(hp_mods)
        das[name] = (ds[var].sel(time=times) * scale).compute()

    fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, layout="constrained",
                             gridspec_kw={"height_ratios": [2, 1]})
    curves = {}
    for name, da in das.items():
        zm, lat = zonal_mean(da)
        curves[name] = zm
        axes[0].plot(lat, zm, label=name)
    axes[0].legend()
    axes[0].set_ylabel(units or das["control"].attrs.get("units", var))
    axes[0].set_title(f"{var}: zonal mean, {days} days from {DAY}")
    axes[1].plot(lat, curves["+4K"] - curves["control"], color="k")
    axes[1].axhline(0, color="0.7", lw=0.8)
    axes[1].set_ylabel("+4K - control")
    axes[1].set_xlabel("latitude")
    axes[1].set_xlim(-90, 90)
    return fig


compare_zonal("tas");

In [ ]:
compare_zonal("pr", scale=86400, units="mm day-1");

## Global means through the year

Zoom 0 is 12 cells, so a whole year of global means loads in seconds — a cheap way to see the response and
check for drift.


In [ ]:
z0 = {name: cat[sim](zoom=0, time="PT1H").to_dask().pipe(hp_mods)
      for name, sim in [("control", CONTROL), ("+4K", P4K)]}

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, layout="constrained")
for var, ax, scale, label in [("tas", axes[0], 1.0, "tas (K)"),
                              ("pr", axes[1], 86400, "pr (mm/day)")]:
    for name, ds in z0.items():
        series = (ds[var].mean(dim="cell") * scale).compute()
        series.rolling(time=24, center=True).mean().plot(ax=ax, label=f"{name} (24h mean)")
    ax.set_ylabel(label)
    ax.set_title("")
    ax.legend()
axes[0].set_title("Global mean, control vs +4K");

In [ ]:
# How big is the response, and how much of it is the ocean?
for var, scale, units in [("tas", 1.0, "K"), ("pr", 86400, "mm/day"), ("rlut", 1.0, "W/m2")]:
    vals = {name: float((ds[var].mean(dim="cell") * scale).mean().compute()) for name, ds in z0.items()}
    diff = vals["+4K"] - vals["control"]
    print(f"{var:>5}: control {vals['control']:9.3f}  +4K {vals['+4K']:9.3f}  "
          f"difference {diff:+8.3f} {units} ({100 * diff / vals['control']:+.1f}%)")

## Vertical structure

The 3-hourly store has variables on pressure levels: how far up does the warming reach, and what happens to
humidity and cloud?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), layout="constrained")
times = slice(DAY, str(pd.Timestamp(DAY) + pd.Timedelta(days=2)))
for var, ax in zip(["ta", "hus", "clw"], axes):
    profiles = {}
    for name, ds in ds3h.items():
        profiles[name] = ds[var].sel(time=times).mean(dim=["time", "cell"]).compute()
        profiles[name].plot(ax=ax, y="pressure", yincrease=False, label=name)
    ax.set_title(f"{var} ({ds3h['control'][var].attrs.get('units', '')})")
    ax.legend()

    twin = ax.twiny()
    (profiles["+4K"] - profiles["control"]).plot(ax=twin, y="pressure", yincrease=False,
                                                 color="k", ls="--")
    twin.set_xlabel("+4K - control", color="k")
    twin.set_title("")
fig.suptitle(f"Global mean profiles, 3 days from {DAY}");

## Things worth trying next

* **Tropical means only** — mask to +/-20 degrees to isolate the convective response from the extratropics.
* **Precipitation extremes**: compare distributions at high zoom rather than means; the mean response and the
  extreme response to warming are not the same thing.
* **Clausius-Clapeyron**: column water vapour (`prw`) should rise roughly 7%/K; does it, and does precipitation
  follow or lag it?
* **Diurnal cycle** of precipitation over land vs ocean, using `groupby("time.hour")`.
* Track features (see `08_demo_mcs_track_stats_healpix_UM_hk26.ipynb`) in both runs and compare statistics.
